# HTR Gratuito - Celorico da Beira (CHURRO-3B)Processa 4152 imagens de registos de óbitos via CHURRO-3B (open-weight, maIs preciso que Gemini).## 1. Instalar CHURRO OCR

In [ ]:
!pip install -q "churro-ocr[hf]" transformers accelerate bitsandbytesprint('CHURRO instalado!')

## 2. Fazer upload das imagens do servidorNo servidor: `cd /home/pxtkhw/projetos/obitos && tar czf full_images.tar.gz output/full_images/`Depois faz upload do `full_images.tar.gz` aqui:

In [ ]:
from google.colab import filesimport osprint('Faz upload do full_images.tar.gz...')# uploaded = files.upload()  # Descomenta para fazer uploados.makedirs('full_images', exist_ok=True)!tar xzf full_images.tar.gz -C . 2>/dev/null || echo 'Faz upload primeiro!'print('Imagens disponíveis:', len([f for f in os.listdir('full_images') if f.endswith('.tiff')]))

## 3. Testar CHURRO-3B (modelo open-weight)CHURRO-3B: https://huggingface.co/stanford-oval/churro-3BPrecisão: superior ao Gemini 2.5 Pro, 15.5x mais barato.

In [ ]:
import osos.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Usa GPU T4 (grátis)try:    from churro_ocr import ChurroOCR    print('A carregar CHURRO-3B...')    ocr = ChurroOCR(backend='hf', model='stanford-oval/churro-3B')    print('CHURRO-3B carregado!')    # Teste com uma imagem    img_path = 'full_images/45283239.tiff'    if os.path.exists(img_path):        result = ocr.transcribe(img_path)        print('CHURRO OUTPUT (primeiros 500 chars):')        print(result.text[:500])    else:        print('Imagem não encontrada. Faz upload primeiro.')except Exception as e:    print(f'Erro: {e}')

## 4. Processar TODAS as imagens (4152)Usa CHURRO-3B para transcrever todas as páginas.

In [ ]:
import os, json, timeos.makedirs('htr_output', exist_ok=True)# Lista de imagensimages = sorted([os.path.join('full_images', f) for f in os.listdir('full_images') if f.endswith('.tiff')])print(f'Total imagens: {len(images)}')# Teste com 10 primeirasfor i, img_path in enumerate(images[:10]):    try:        result = ocr.transcribe(img_path)        text = result.text                # Guardar resultado        output = {'raw_text': text, 'model': 'churro-3B'}        output_file = os.path.join('htr_output', os.path.basename(img_path).replace('.tiff', '.json'))        with open(output_file, 'w') as f:            json.dump(output, f, indent=2, ensure_ascii=False)                print(f'[{i+1}/10] {os.path.basename(img_path)}: OK ({len(text)} chars)')        time.sleep(1)  # Pausa entre requests    except Exception as e:        print(f'[{i+1}/10] {os.path.basename(img_path)}: Erro - {e}')print('Teste completo! Remove [:10] para processar todas.')

## 5. Fazer download dos resultadosGera `htr_output.tar.gz` para fazer download.

In [ ]:
import tarfilefrom google.colab import filesprint('A criar htr_output.tar.gz...')with tarfile.open('htr_output.tar.gz', 'w:gz') as tar:    tar.add('htr_output')print('Download do ficheiro...')files.download('htr_output.tar.gz')print('Download pronto!')

## 6. Alternativa: TrOCR (Português)Se CHURRO falhar, usa TrOCR fine-tuned para português:

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModelfrom PIL import Imageimport torchprint('A carregar TrOCR para português...')# Modelo base: microsoft/trocr-base-printed (substituir por modelo português se disponível)processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')device = 'cuda' if torch.cuda.is_available() else 'cpu'model.to(device)print(f'TrOCR carregado (device: {device})')def recognize_text(image_path):    image = Image.open(image_path).convert('RGB')    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)    generated_ids = model.generate(pixel_values)    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]if os.path.exists('full_images/45283239.tiff'):    text = recognize_text('full_images/45283239.tiff')    print('TrOCR OUTPUT:')    print(text[:500])